# loading


In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
BASE_PATH = "D:/..amikom/6th/data-mining-project"
cbf_df  = pd.read_csv(f"{BASE_PATH}/cbf_final.csv", index_col=0)
ratings = pd.read_csv(f"{BASE_PATH}/goodbooks-10k/ratings.csv")
books   = pd.read_csv(f"{BASE_PATH}/books_clean.csv", index_col=0)

# cb with word2vec


In [3]:
cbf_df.head()

,id,book_id,title,authors,decade_year,tag_name,tag_length
0,1,2767052,"The Hunger Games (The Hunger Games, #1)",author_suzanne_collins,decade_2000s,"young_adult, fiction, dystopian, dystopia, fan...",100
1,2,3,Harry Potter and the Sorcerer's Stone (Harry P...,author_jk_rowling author_mary_grandpr,decade_1990s,"fantasy, young_adult, fiction, harry_potter, s...",96
2,3,41865,"Twilight (Twilight, #1)",author_stephenie_meyer,decade_2000s,"young_adult, fantasy, vampire, fiction, parano...",109
3,4,2657,To Kill a Mockingbird,author_harper_lee,decade_1960s,"classic, classic, historical_fiction, school, ...",107
4,5,4671,The Great Gatsby,author_f_scott_fitzgerald,decade_1920s,"classic, fiction, classic, literature, school,...",115


In [4]:
cbf_df.drop(columns=['book_id'], inplace=True)
cbf_df.rename(columns={'id': 'book_id'}, inplace=True)
cbf_df.head()

,book_id,title,authors,decade_year,tag_name,tag_length
0,1,"The Hunger Games (The Hunger Games, #1)",author_suzanne_collins,decade_2000s,"young_adult, fiction, dystopian, dystopia, fan...",100
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,author_jk_rowling author_mary_grandpr,decade_1990s,"fantasy, young_adult, fiction, harry_potter, s...",96
2,3,"Twilight (Twilight, #1)",author_stephenie_meyer,decade_2000s,"young_adult, fantasy, vampire, fiction, parano...",109
3,4,To Kill a Mockingbird,author_harper_lee,decade_1960s,"classic, classic, historical_fiction, school, ...",107
4,5,The Great Gatsby,author_f_scott_fitzgerald,decade_1920s,"classic, fiction, classic, literature, school,...",115


In [5]:
cbf_df['tag_name_clean'] = cbf_df['tag_name'].str.replace(',', ' ')

cbf_df['soup'] = cbf_df['authors'] + " " + cbf_df['decade_year'] + " " + cbf_df['tag_name_clean']

tokenized_soup = cbf_df['soup'].apply(lambda x: x.split())

w2v_model = Word2Vec(
    sentences=tokenized_soup,
    vector_size=100, window=5, min_count=1, workers=4, epochs=10
)

def get_book_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

book_vectors = np.array([get_book_vector(t, w2v_model) for t in tokenized_soup])
cbf_sim_matrix = cosine_similarity(book_vectors)

# Lookup maps
cbf_title_to_idx = {title: i for i, title in enumerate(cbf_df['title'])}
cbf_id_to_idx    = {bid: i   for i, bid   in enumerate(cbf_df['book_id'])}

In [6]:
cbf_df.head()

,book_id,title,authors,decade_year,tag_name,tag_length,tag_name_clean,soup
0,1,"The Hunger Games (The Hunger Games, #1)",author_suzanne_collins,decade_2000s,"young_adult, fiction, dystopian, dystopia, fan...",100,young_adult fiction dystopian dystopia fan...,author_suzanne_collins decade_2000s young_adul...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,author_jk_rowling author_mary_grandpr,decade_1990s,"fantasy, young_adult, fiction, harry_potter, s...",96,fantasy young_adult fiction harry_potter s...,author_jk_rowling author_mary_grandpr decade_1...
2,3,"Twilight (Twilight, #1)",author_stephenie_meyer,decade_2000s,"young_adult, fantasy, vampire, fiction, parano...",109,young_adult fantasy vampire fiction parano...,author_stephenie_meyer decade_2000s young_adul...
3,4,To Kill a Mockingbird,author_harper_lee,decade_1960s,"classic, classic, historical_fiction, school, ...",107,classic classic historical_fiction school ...,author_harper_lee decade_1960s classic classi...
4,5,The Great Gatsby,author_f_scott_fitzgerald,decade_1920s,"classic, fiction, classic, literature, school,...",115,classic fiction classic literature school ...,author_f_scott_fitzgerald decade_1920s classic...


In [7]:
tokenized_soup

0       [author_suzanne_collins, decade_2000s, young_a...
1       [author_jk_rowling, author_mary_grandpr, decad...
2       [author_stephenie_meyer, decade_2000s, young_a...
3       [author_harper_lee, decade_1960s, classic, cla...
4       [author_f_scott_fitzgerald, decade_1920s, clas...
                              ...                        
8725    [author_herman_melville, decade_1920s, classic...
8726    [author_ilona_andrews, decade_2010s, urban_fan...
8727    [author_robert_a_caro, decade_1990s, biography...
8728    [author_patrick_obrian, decade_1970s, historic...
8729    [author_peggy_orenstein, decade_2010s, non_fic...
Name: soup, Length: 8730, dtype: object

In [8]:
book_vectors

array([[-0.23554282, -0.28430393,  0.9058518 , ...,  0.3080037 ,
         0.5854063 , -0.8273792 ],
       [-0.18468864, -0.12509336,  0.589062  , ...,  0.655989  ,
        -0.05979095, -0.15672   ],
       [-0.13891429, -0.49481273,  1.0352036 , ...,  0.8564809 ,
         0.45365545, -1.6935902 ],
       ...,
       [-0.18552423,  1.6020869 , -0.29409328, ..., -0.26541606,
        -0.32010373,  0.29811925],
       [-0.2607378 ,  0.6316968 ,  0.51601434, ...,  0.13679378,
        -0.01036694,  0.29398617],
       [-0.1934042 ,  1.0890583 , -0.23797621, ..., -0.11983091,
         0.09639737, -0.30895343]], dtype=float32)

In [9]:
cbf_sim_matrix

array([[0.99999994, 0.85390836, 0.8184175 , ..., 0.16171391, 0.5882348 ,
        0.26378044],
       [0.85390836, 0.9999998 , 0.72463197, ..., 0.2676177 , 0.6913161 ,
        0.2782469 ],
       [0.8184175 , 0.72463197, 1.0000001 , ..., 0.1394842 , 0.3665318 ,
        0.29730642],
       ...,
       [0.16171391, 0.2676177 , 0.1394842 , ..., 0.99999994, 0.650869  ,
        0.9298641 ],
       [0.5882348 , 0.6913161 , 0.3665318 , ..., 0.650869  , 0.9999999 ,
        0.54618526],
       [0.26378044, 0.2782469 , 0.29730642, ..., 0.9298641 , 0.54618526,
        1.        ]], dtype=float32)

In [10]:
cbf_title_to_idx

{'The Hunger Games (The Hunger Games, #1)': 0,
 "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)": 1,
 'Twilight (Twilight, #1)': 2,
 'To Kill a Mockingbird': 3,
 'The Great Gatsby': 4,
 'The Fault in Our Stars': 5,
 'The Hobbit': 6,
 'The Catcher in the Rye': 7,
 'Angels & Demons  (Robert Langdon, #1)': 8,
 'Pride and Prejudice': 9,
 'The Kite Runner': 10,
 'Divergent (Divergent, #1)': 11,
 '1984': 12,
 'Animal Farm': 13,
 'The Diary of a Young Girl': 14,
 'The Girl with the Dragon Tattoo (Millennium, #1)': 15,
 'Catching Fire (The Hunger Games, #2)': 16,
 'Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)': 17,
 'The Fellowship of the Ring (The Lord of the Rings, #1)': 18,
 'Mockingjay (The Hunger Games, #3)': 19,
 'Harry Potter and the Order of the Phoenix (Harry Potter, #5)': 20,
 'The Lovely Bones': 21,
 'Harry Potter and the Chamber of Secrets (Harry Potter, #2)': 22,
 'Harry Potter and the Goblet of Fire (Harry Potter, #4)': 23,
 'Harry Potter and the Deathly 

In [11]:
cbf_id_to_idx

{1: 0,
 2: 1,
 3: 2,
 4: 3,
 5: 4,
 6: 5,
 7: 6,
 8: 7,
 9: 8,
 10: 9,
 11: 10,
 12: 11,
 13: 12,
 14: 13,
 15: 14,
 16: 15,
 17: 16,
 18: 17,
 19: 18,
 20: 19,
 21: 20,
 22: 21,
 23: 22,
 24: 23,
 25: 24,
 26: 25,
 27: 26,
 28: 27,
 29: 28,
 30: 29,
 31: 30,
 32: 31,
 33: 32,
 34: 33,
 35: 34,
 36: 35,
 37: 36,
 38: 37,
 39: 38,
 40: 39,
 41: 40,
 42: 41,
 43: 42,
 44: 43,
 46: 44,
 47: 45,
 49: 46,
 50: 47,
 51: 48,
 52: 49,
 53: 50,
 54: 51,
 55: 52,
 56: 53,
 57: 54,
 58: 55,
 59: 56,
 60: 57,
 61: 58,
 62: 59,
 63: 60,
 64: 61,
 65: 62,
 66: 63,
 67: 64,
 68: 65,
 69: 66,
 70: 67,
 71: 68,
 72: 69,
 73: 70,
 74: 71,
 75: 72,
 76: 73,
 77: 74,
 78: 75,
 79: 76,
 80: 77,
 81: 78,
 82: 79,
 83: 80,
 85: 81,
 86: 82,
 87: 83,
 88: 84,
 89: 85,
 90: 86,
 91: 87,
 92: 88,
 93: 89,
 94: 90,
 95: 91,
 96: 92,
 97: 93,
 98: 94,
 99: 95,
 100: 96,
 101: 97,
 102: 98,
 103: 99,
 104: 100,
 105: 101,
 106: 102,
 107: 103,
 108: 104,
 109: 105,
 110: 106,
 111: 107,
 112: 108,
 113: 109,
 114:

# cf with svd++ from surprise


In [12]:
books

,id,book_id,title,authors,decade_year
0,1,2767052,"The Hunger Games (The Hunger Games, #1)",author_suzanne_collins,decade_2000s
1,2,3,Harry Potter and the Sorcerer's Stone (Harry P...,author_jk_rowling author_mary_grandpr,decade_1990s
2,3,41865,"Twilight (Twilight, #1)",author_stephenie_meyer,decade_2000s
3,4,2657,To Kill a Mockingbird,author_harper_lee,decade_1960s
4,5,4671,The Great Gatsby,author_f_scott_fitzgerald,decade_1920s
...,...,...,...,...,...
9994,9995,15613,"Billy Budd, Sailor",author_herman_melville,decade_1920s
9995,9996,7130616,"Bayou Moon (The Edge, #2)",author_ilona_andrews,decade_2010s
9996,9997,208324,"Means of Ascent (The Years of Lyndon Johnson, #2)",author_robert_a_caro,decade_1990s
9997,9998,77431,The Mauritius Command,author_patrick_obrian,decade_1970s


In [13]:
books.drop(columns=['book_id'], inplace=True)
books.rename(columns={'id': 'book_id'}, inplace=True)
books.tail()

,book_id,title,authors,decade_year
9994,9995,"Billy Budd, Sailor",author_herman_melville,decade_1920s
9995,9996,"Bayou Moon (The Edge, #2)",author_ilona_andrews,decade_2010s
9996,9997,"Means of Ascent (The Years of Lyndon Johnson, #2)",author_robert_a_caro,decade_1990s
9997,9998,The Mauritius Command,author_patrick_obrian,decade_1970s
9998,9999,Cinderella Ate My Daughter: Dispatches from th...,author_peggy_orenstein,decade_2010s


In [14]:
ratings

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [15]:
# Filter to only book_ids present in the books metadata file
valid_ids        = set(books['book_id'].unique()) if 'book_id' in books.columns else set(books['book_id'].unique())
ratings_filtered = ratings[ratings['book_id'].isin(valid_ids)].copy()

In [16]:
len(valid_ids)

8730

In [17]:
ratings_filtered['book_id'].nunique()

8730

In [18]:
# Build Surprise dataset
reader   = Reader(rating_scale=(1, 5))
data     = Dataset.load_from_df(ratings_filtered[['user_id', 'book_id', 'rating']], reader)
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

In [19]:
reader

In [20]:
data

In [21]:
trainset

In [22]:
testset

[(14001, 5306, 4.0),
 (40226, 7111, 4.0),
 (38300, 4012, 3.0),
 (34189, 2509, 3.0),
 (49505, 7834, 5.0),
 (50976, 404, 4.0),
 (9135, 6650, 3.0),
 (29282, 4625, 5.0),
 (24461, 6592, 5.0),
 (5569, 5101, 5.0),
 (15687, 805, 3.0),
 (37944, 8561, 5.0),
 (41495, 1049, 5.0),
 (21954, 8572, 4.0),
 (6430, 9666, 3.0),
 (8209, 3715, 2.0),
 (52051, 7285, 5.0),
 (11837, 442, 3.0),
 (4900, 1052, 3.0),
 (11957, 9877, 5.0),
 (23767, 7591, 4.0),
 (8958, 7747, 5.0),
 (15157, 2732, 3.0),
 (17144, 4727, 5.0),
 (39213, 4148, 4.0),
 (47232, 6486, 5.0),
 (18077, 2329, 4.0),
 (11325, 3958, 5.0),
 (10231, 1965, 5.0),
 (6016, 1630, 3.0),
 (39177, 3852, 4.0),
 (24437, 4850, 5.0),
 (49050, 5624, 3.0),
 (31537, 8314, 5.0),
 (10297, 167, 5.0),
 (19484, 334, 3.0),
 (7064, 5048, 4.0),
 (3521, 6488, 4.0),
 (22384, 4519, 2.0),
 (4715, 259, 3.0),
 (30554, 7265, 3.0),
 (15567, 3767, 2.0),
 (36516, 3024, 5.0),
 (4383, 4905, 2.0),
 (19729, 808, 4.0),
 (27687, 8250, 3.0),
 (41888, 5732, 5.0),
 (5629, 1954, 3.0),
 (13544, 28

In [23]:
# Keep plain DataFrames for evaluation helpers that need them
train_df = ratings_filtered.sample(frac=0.8, random_state=42)
test_df  = ratings_filtered.drop(train_df.index)

In [24]:
train_df

,book_id,user_id,rating
224336,2245,39445,5
390442,3910,25368,4
217756,2179,48234,5
771881,7779,39520,3
884748,8957,37947,4
...,...,...,...
335889,3363,16821,5
891153,9026,2182,3
351482,3519,47968,5
255984,2562,33982,5


In [25]:
test_df

,book_id,user_id,rating
3,1,1169,4
5,1,2077,4
8,1,3662,4
13,1,6630,5
15,1,9246,1
...,...,...,...
981640,9999,33261,2
981642,9999,35005,3
981653,9999,42893,4
981654,9999,45918,5


In [26]:
# Train Surprise SVD
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

In [27]:
svd_model

In [28]:
# Evaluate with Surprise's built-in metrics
predictions = svd_model.test(testset)
print(f"CF Test RMSE : {accuracy.rmse(predictions, verbose=False):.4f}")
print(f"CF Test MAE  : {accuracy.mae(predictions,  verbose=False):.4f}")

CF Test RMSE : 0.8439
CF Test MAE  : 0.6606


In [29]:
# Pre-cache: all unique book IDs known to the CF model (inner ids)
cf_all_book_ids = list(ratings_filtered['book_id'].unique())

In [30]:
cf_all_book_ids

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 46,
 47,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 

## mlflow tracking


In [31]:
param_grid = {
    'n_factors':  [50, 100, 150],      # size of latent factor space
    'n_epochs':   [10, 20],            # SGD passes over the training data
    'lr_all':     [0.002, 0.005],      # learning rate for all parameters
    'reg_all':    [0.02, 0.1],         # L2 regularisation strength
}

In [32]:
import itertools

keys   = list(param_grid.keys())
combos = [dict(zip(keys, vals)) for vals in itertools.product(*param_grid.values())]
print(f"Running {len(combos)} experiments...\n")

Running 24 experiments...



In [33]:
import mlflow
import mlflow.sklearn

EXPERIMENT_NAME = "SVD_Collaborative_Filtering"
mlflow.set_experiment(EXPERIMENT_NAME)
 
best_rmse  = float('inf')
best_run_id = None
best_model  = None
best_params = None

In [34]:
for i, params in enumerate(combos, 1):
    run_name = (
        f"svd_f{params['n_factors']}_e{params['n_epochs']}"
        f"_lr{params['lr_all']}_reg{params['reg_all']}"
    )
    print(f"[{i:>2}/{len(combos)}] {run_name}")
 
    with mlflow.start_run(run_name=run_name):
 
        # ── Log hyperparameters ───────────────────────────────────────────
        mlflow.log_params(params)
        mlflow.log_param("test_size",    0.2)
        mlflow.log_param("random_state", 42)
 
        # ── Train ─────────────────────────────────────────────────────────
        model = SVD(
            n_factors = params['n_factors'],
            n_epochs  = params['n_epochs'],
            lr_all    = params['lr_all'],
            reg_all   = params['reg_all'],
            random_state = 42,
        )
        model.fit(trainset)
 
        # ── Evaluate ──────────────────────────────────────────────────────
        predictions = model.test(testset)
        rmse = accuracy.rmse(predictions, verbose=False)
        mae  = accuracy.mae(predictions,  verbose=False)
 
        # Coverage: fraction of (user, item) pairs the model can predict
        # (always 1.0 for SVD, useful baseline to log for other model types)
        n_users = trainset.n_users
        n_items = trainset.n_items
 
        # Bias magnitudes — useful for diagnosing over/underfitting
        user_bias_mean = float(np.abs(model.bu).mean())
        item_bias_mean = float(np.abs(model.bi).mean())
 
        # ── Log metrics ───────────────────────────────────────────────────
        mlflow.log_metrics({
            "rmse":            round(rmse, 4),
            "mae":             round(mae,  4),
            "n_users":         n_users,
            "n_items":         n_items,
            "user_bias_mean":  round(user_bias_mean, 4),
            "item_bias_mean":  round(item_bias_mean, 4),
        })
 
        # ── Log model artifact (best run only to save disk space) ─────────
        run_id = mlflow.active_run().info.run_id
        if rmse < best_rmse:
            best_rmse   = rmse
            best_run_id = run_id
            best_model  = model
            best_params = params
            mlflow.log_param("is_best", True)
        else:
            mlflow.log_param("is_best", False)
 
        print(f"         RMSE={rmse:.4f}  MAE={mae:.4f}")

[ 1/24] svd_f50_e10_lr0.002_reg0.02
         RMSE=0.8690  MAE=0.6904
[ 2/24] svd_f50_e10_lr0.002_reg0.1
         RMSE=0.8697  MAE=0.6916
[ 3/24] svd_f50_e10_lr0.005_reg0.02
         RMSE=0.8479  MAE=0.6676
[ 4/24] svd_f50_e10_lr0.005_reg0.1
         RMSE=0.8474  MAE=0.6690
[ 5/24] svd_f50_e20_lr0.002_reg0.02
         RMSE=0.8518  MAE=0.6722
[ 6/24] svd_f50_e20_lr0.002_reg0.1
         RMSE=0.8517  MAE=0.6736
[ 7/24] svd_f50_e20_lr0.005_reg0.02
         RMSE=0.8404  MAE=0.6575
[ 8/24] svd_f50_e20_lr0.005_reg0.1
         RMSE=0.8383  MAE=0.6585
[ 9/24] svd_f100_e10_lr0.002_reg0.02
         RMSE=0.8721  MAE=0.6932
[10/24] svd_f100_e10_lr0.002_reg0.1
         RMSE=0.8720  MAE=0.6937
[11/24] svd_f100_e10_lr0.005_reg0.02
         RMSE=0.8511  MAE=0.6704
[12/24] svd_f100_e10_lr0.005_reg0.1
         RMSE=0.8490  MAE=0.6704
[13/24] svd_f100_e20_lr0.002_reg0.02
         RMSE=0.8549  MAE=0.6751
[14/24] svd_f100_e20_lr0.002_reg0.1
         RMSE=0.8534  MAE=0.6752
[15/24] svd_f100_e20_lr0.005_reg0.0

In [35]:
import pickle

print(f"\n{'='*55}")
print(f"  Best run  : {best_run_id}")
print(f"  Best RMSE : {best_rmse:.4f}")
print(f"  Params    : {best_params}")
print(f"{'='*55}\n")
 
# Log the best model artifact back into its own run
with mlflow.start_run(run_id=best_run_id):
    # Save with pickle so trainset is preserved (avoids score drift on reload)
    with open('svd_best_model.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    mlflow.log_artifact('svd_best_model.pkl', artifact_path='model')
    print("Best model saved to 'svd_best_model.pkl' and logged as MLflow artifact.")


  Best run  : a3cb5ca4de3144db93fc991d84b75cec
  Best RMSE : 0.8383
  Params    : {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.1}

Best model saved to 'svd_best_model.pkl' and logged as MLflow artifact.


In [36]:
client  = mlflow.tracking.MlflowClient()
exp     = client.get_experiment_by_name(EXPERIMENT_NAME)
runs    = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"]
)
 
rows = []
for r in runs:
    p = r.data.params
    m = r.data.metrics
    rows.append({
        "run_name":   r.info.run_name,
        "n_factors":  p.get("n_factors"),
        "n_epochs":   p.get("n_epochs"),
        "lr_all":     p.get("lr_all"),
        "reg_all":    p.get("reg_all"),
        "RMSE":       round(m.get("rmse", 0), 4),
        "MAE":        round(m.get("mae",  0), 4),
        "is_best":    p.get("is_best"),
    })
 
summary = pd.DataFrame(rows)
print("\nAll runs (sorted by RMSE):")
print(summary.to_string(index=False))
print(f"\nRun `mlflow ui` to explore results in the browser.")


All runs (sorted by RMSE):
                    run_name n_factors n_epochs lr_all reg_all   RMSE    MAE is_best
  svd_f50_e20_lr0.005_reg0.1        50       20  0.005     0.1 0.8383 0.6585    True
  svd_f50_e20_lr0.005_reg0.1        50       20  0.005     0.1 0.8383 0.6585    True
 svd_f100_e20_lr0.005_reg0.1       100       20  0.005     0.1 0.8389 0.6593   False
 svd_f100_e20_lr0.005_reg0.1       100       20  0.005     0.1 0.8389 0.6593   False
 svd_f150_e20_lr0.005_reg0.1       150       20  0.005     0.1 0.8397 0.6601   False
 svd_f150_e20_lr0.005_reg0.1       150       20  0.005     0.1 0.8397 0.6601   False
 svd_f50_e20_lr0.005_reg0.02        50       20  0.005    0.02 0.8404 0.6575    True
 svd_f50_e20_lr0.005_reg0.02        50       20  0.005    0.02 0.8404 0.6575    True
svd_f100_e20_lr0.005_reg0.02       100       20  0.005    0.02 0.8439 0.6606   False
svd_f100_e20_lr0.005_reg0.02       100       20  0.005    0.02 0.8439 0.6606   False
svd_f150_e20_lr0.005_reg0.02       15

# hybrid


## functions


In [3]:
def get_cf_scores(user_id):
    """
    Return {book_id: normalized_cf_score} using Surprise SVD predictions.
    Surprise predicts on the 1-5 scale; we normalize to [0, 1] for blending.
    Unknown users return an empty dict (triggers CBF fallback).
    """
    # Check if user exists in the trainset
    try:
        svd_model.trainset.to_inner_uid(user_id)
    except ValueError:
        return {}

    raw_scores = {
        bid: svd_model.predict(user_id, bid).est
        for bid in cf_all_book_ids
    }
    vals   = np.array(list(raw_scores.values()))
    mn, mx = vals.min(), vals.max()
    normed = (vals - mn) / (mx - mn) if mx != mn else np.zeros_like(vals)
    return {bid: float(normed[i]) for i, bid in enumerate(raw_scores)}

In [4]:
def get_cbf_scores(title):
    """Return {book_id: normalized_cbf_score} for every CBF book."""
    if title not in cbf_title_to_idx:
        return {}
    idx    = cbf_title_to_idx[title]
    sims   = cbf_sim_matrix[idx]
    mn, mx = sims.min(), sims.max()
    normed = (sims - mn) / (mx - mn) if mx != mn else np.zeros_like(sims)
    return {cbf_df.iloc[i]['book_id']: normed[i] for i in range(len(normed))}

In [5]:
def recommend_hybrid(user_id, seed_title, n=10, alpha=0.5):
    """
    Blend CBF (Word2Vec cosine similarity) and CF (SVD) recommendations.

    Parameters
    ----------
    user_id    : int   – target user
    seed_title : str   – anchor book for content similarity
    n          : int   – number of results to return
    alpha      : float – CF weight (0 = pure CBF, 1 = pure CF)

    Returns
    -------
    DataFrame [Title, Authors, CF_Score, CBF_Score, Hybrid_Score]
    """
    cf_scores  = get_cf_scores(user_id)
    cbf_scores = get_cbf_scores(seed_title)

    if not cf_scores and not cbf_scores:
        return "User ID and book title both not found."
    if not cf_scores:
        print("⚠️  User not found — falling back to pure CBF.")
        alpha = 0.0
    if not cbf_scores:
        print("⚠️  Seed title not found — falling back to pure CF.")
        alpha = 1.0

    read_book_ids = set(ratings_filtered[ratings_filtered['user_id'] == user_id]['book_id'])
    all_book_ids  = set(cf_scores) | set(cbf_scores)

    results = []
    for bid in all_book_ids:
        if bid in read_book_ids:
            continue

        cf_s   = cf_scores.get(bid, 0.0)
        cbf_s  = cbf_scores.get(bid, 0.0)
        hybrid = alpha * cf_s + (1 - alpha) * cbf_s

        book_row = books[books['book_id'] == bid]
        if book_row.empty:
            book_row = books[books['id'] == bid]
        if book_row.empty:
            continue

        results.append({
            "Title":        book_row['title'].values[0],
            "Authors":      book_row['authors'].values[0],
            "CF_Score":     round(cf_s,   4),
            "CBF_Score":    round(cbf_s,  4),
            "Hybrid_Score": round(hybrid, 4),
        })

    if not results:
        return "No recommendations found."

    result_df = (
        pd.DataFrame(results)
        .sort_values("Hybrid_Score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    result_df.index += 1
    return result_df

## evaluation


In [6]:
def _get_hybrid_ranked_list(user_id, seed_title, alpha, n_candidates=100):
    """
    Internal helper: returns a ranked list of book_ids for evaluation.
    Does NOT filter already-read books (that's handled by the test split).
    """
    cf_scores  = get_cf_scores(user_id)
    cbf_scores = get_cbf_scores(seed_title)
    if not cf_scores and not cbf_scores:
        return []

    eff_alpha = alpha
    if not cf_scores:  eff_alpha = 0.0
    if not cbf_scores: eff_alpha = 1.0

    all_book_ids = set(cf_scores) | set(cbf_scores)
    scored = [
        (bid, eff_alpha * cf_scores.get(bid, 0.0) +
              (1 - eff_alpha) * cbf_scores.get(bid, 0.0))
        for bid in all_book_ids
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [bid for bid, _ in scored[:n_candidates]]


def ndcg_at_k(recommended, relevant_set, k):
    """
    Normalised Discounted Cumulative Gain @ k.

    Uses binary relevance (1 if book is in relevant_set, else 0).
    IDCG assumes all relevant items are placed at the top ranks.
    """
    dcg = sum(
        1.0 / np.log2(rank + 1)
        for rank, bid in enumerate(recommended[:k], start=1)
        if bid in relevant_set
    )
    n_relevant = min(len(relevant_set), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, n_relevant + 1))
    return dcg / idcg if idcg > 0 else 0.0


def precision_at_k(recommended, relevant_set, k):
    """Fraction of top-k recommendations that are relevant."""
    hits = sum(1 for bid in recommended[:k] if bid in relevant_set)
    return hits / k


def recall_at_k(recommended, relevant_set, k):
    """Fraction of all relevant items that appear in top-k."""
    if not relevant_set:
        return 0.0
    hits = sum(1 for bid in recommended[:k] if bid in relevant_set)
    return hits / len(relevant_set)


def hit_rate_at_k(recommended, relevant_set, k):
    """1 if at least one relevant item is in top-k, else 0."""
    return 1.0 if any(bid in relevant_set for bid in recommended[:k]) else 0.0


def average_precision_at_k(recommended, relevant_set, k):
    """
    Average Precision @ k.
    AP = mean of precision@i for every rank i where item i is relevant.
    Rewards systems that place relevant items earlier in the list.
    """
    hits, score = 0, 0.0
    for rank, bid in enumerate(recommended[:k], start=1):
        if bid in relevant_set:
            hits += 1
            score += hits / rank
    return score / min(len(relevant_set), k) if relevant_set else 0.0


def evaluate_hybrid(alpha=0.5, k_values=(5, 10, 20),
                    relevance_threshold=4, n_test_users=200,
                    min_test_relevant=2, random_state=42):
    """
    Evaluate the hybrid recommender on the held-out test split.

    Strategy
    --------
    For each sampled test user:
      - Ground truth = their test-set books rated >= relevance_threshold.
      - CBF seed     = their highest-rated book in the training split.
      - Rank all candidate books with the hybrid, compute metrics.

    Parameters
    ----------
    alpha               : float – CF weight (0=pure CBF, 1=pure CF)
    k_values            : tuple – cut-off ranks to evaluate
    relevance_threshold : int   – min rating to count as "liked"
    n_test_users        : int   – users to sample (speed vs. accuracy)
    min_test_relevant   : int   – skip users with fewer relevant test items
    random_state        : int   – reproducible sampling

    Returns
    -------
    summary   : dict  {metric: {k: mean_score}}
    detail_df : DataFrame with per-user scores (for deeper inspection)
    """
    rng = np.random.default_rng(random_state)

    train_users = set(train_df['user_id'].unique())
    test_users  = set(test_df['user_id'].unique())
    eligible    = list(train_users & test_users)
    rng.shuffle(eligible)
    eligible    = eligible[:n_test_users]

    # Best training-set book per user (used as CBF seed)
    train_top = (
        train_df.sort_values('rating', ascending=False)
        .groupby('user_id').first().reset_index()
    )[['user_id', 'book_id']]
    uid_to_seed_bid = dict(zip(train_top['user_id'], train_top['book_id']))
    rows = []
    for uid in eligible:
        user_test = test_df[test_df['user_id'] == uid]
        relevant  = set(
            user_test[user_test['rating'] >= relevance_threshold]['book_id']
        )
        if len(relevant) < min_test_relevant:
            continue

        seed_bid   = uid_to_seed_bid.get(uid)
        seed_match = cbf_df[cbf_df['book_id'] == seed_bid]['title']
        seed_title = seed_match.values[0] if not seed_match.empty else ""

        recommended = _get_hybrid_ranked_list(
            uid, seed_title, alpha, n_candidates=500
        )

        row = {"user_id": uid, "n_relevant": len(relevant)}
        for k in k_values:
            row[f"NDCG@{k}"] = ndcg_at_k(recommended, relevant, k)
            row[f"P@{k}"]    = precision_at_k(recommended, relevant, k)
            row[f"R@{k}"]    = recall_at_k(recommended, relevant, k)
            row[f"HR@{k}"]   = hit_rate_at_k(recommended, relevant, k)
            row[f"MAP@{k}"]  = average_precision_at_k(recommended, relevant, k)
        rows.append(row)

    if not rows:
        print("No eligible users found for evaluation.")
        return {}, pd.DataFrame()

    detail_df   = pd.DataFrame(rows)
    metric_cols = [c for c in detail_df.columns if c not in ('user_id', 'n_relevant')]
    means       = detail_df[metric_cols].mean()

    summary = {}
    for k in k_values:
        for metric in ('NDCG', 'P', 'R', 'HR', 'MAP'):
            summary.setdefault(metric, {})[k] = round(means[f"{metric}@{k}"], 4)

    return summary, detail_df
  
def print_evaluation_report(alpha=0.5, k_values=(5, 10, 20),
                             relevance_threshold=4, n_test_users=200):
    """Run evaluation and pretty-print a summary table."""
    print(f"\n{'='*62}")
    print("  HYBRID RECOMMENDER — RANKING EVALUATION REPORT")
    print(f"{'='*62}")
    print(f"  alpha={alpha}  |  threshold≥{relevance_threshold}  |  users={n_test_users}")
    print(f"{'='*62}")

    summary, detail_df = evaluate_hybrid(
        alpha=alpha, k_values=k_values,
        relevance_threshold=relevance_threshold,
        n_test_users=n_test_users
    )
    if not summary:
        return None, None

    k_header = "".join(f"{'@'+str(k):>10}" for k in k_values)
    print(f"\n{'Metric':<12}{k_header}")
    print("-" * (12 + 10 * len(k_values)))

    labels = [('NDCG','NDCG'), ('P','Precision'), ('R','Recall'),
              ('HR','Hit Rate'), ('MAP','MAP')]
    for key, label in labels:
        row = f"{label:<12}"
        for k in k_values:
            row += f"{summary[key][k]:>10.4f}"
        print(row)

    print(f"\nEvaluated on {len(detail_df)} users.")
    print("\nMetric guide:")
    print("  NDCG      – ranking quality; rewards hits ranked higher (0→1, higher=better)")
    print("  Precision – fraction of top-k that are relevant")
    print("  Recall    – fraction of relevant items found in top-k")
    print("  Hit Rate  – % of users who got ≥1 relevant item in top-k")
    print("  MAP       – mean average precision across hit positions")

    return summary, detail_df

## save artifacts


In [42]:
import pickle
import os

ARTIFACTS = {
    'cbf_sim_matrix': 'D:/..amikom/6th/data-mining-project/artifacts/cbf_sim_matrix.npy',
    'cbf_meta':       'D:/..amikom/6th/data-mining-project/artifacts/cbf_meta.pkl',
    'svd_model':      'D:/..amikom/6th/data-mining-project/artifacts/svd_model.pkl',
    'ratings_df':     'D:/..amikom/6th/data-mining-project/artifacts/ratings_filtered.pkl',
    'train_test':     'D:/..amikom/6th/data-mining-project/artifacts/train_test_split.pkl',
}

def save_artifacts():
    """
    Persist all objects needed to serve recommendations without retraining.

    Files written
    -------------
    artifacts/cbf_sim_matrix.npy   - cosine similarity matrix (float32)
    artifacts/cbf_meta.pkl         - CBF lookup dicts + cbf_df
    artifacts/svd_model.pkl        - full Surprise SVD model (incl. trainset)
    artifacts/ratings_filtered.pkl - filtered ratings DataFrame + cf_all_book_ids
    artifacts/train_test_split.pkl - train_df / test_df (for evaluation)
    """
    os.makedirs('D:/..amikom/6th/data-mining-project/artifacts', exist_ok=True)

    # Store as float32 to halve disk size (cosine sim doesn't need float64)
    np.save(ARTIFACTS['cbf_sim_matrix'], cbf_sim_matrix.astype(np.float32))

    with open(ARTIFACTS['cbf_meta'], 'wb') as f:
        pickle.dump({
            'cbf_df':           cbf_df,
            'cbf_title_to_idx': cbf_title_to_idx,
            'cbf_id_to_idx':    cbf_id_to_idx,
        }, f)

    # Plain pickle preserves trainset inside svd_model — avoids score drift on reload
    with open(ARTIFACTS['svd_model'], 'wb') as f:
        pickle.dump(svd_model, f)

    with open(ARTIFACTS['ratings_df'], 'wb') as f:
        pickle.dump({
            'ratings_filtered': ratings_filtered,
            'cf_all_book_ids':  cf_all_book_ids,
        }, f)

    with open(ARTIFACTS['train_test'], 'wb') as f:
        pickle.dump({'train_df': train_df, 'test_df': test_df}, f)

    print("Artifacts saved:")
    for name, path in ARTIFACTS.items():
        if os.path.exists(path):
            print(f"  {path:<45} {os.path.getsize(path)/1e6:.1f} MB")

save_artifacts()


Artifacts saved:
  D:/..amikom/6th/data-mining-project/artifacts/cbf_sim_matrix.npy 304.9 MB
  D:/..amikom/6th/data-mining-project/artifacts/cbf_meta.pkl 4.1 MB
  D:/..amikom/6th/data-mining-project/artifacts/svd_model.pkl 68.1 MB
  D:/..amikom/6th/data-mining-project/artifacts/ratings_filtered.pkl 27.7 MB
  D:/..amikom/6th/data-mining-project/artifacts/train_test_split.pkl 27.5 MB


## load artifacts (use this in a new session instead of retraining)


In [7]:
import pickle, os
import numpy as np

ARTIFACTS = {
    'cbf_sim_matrix': 'D:/..amikom/6th/data-mining-project/artifacts/cbf_sim_matrix.npy',
    'cbf_meta':       'D:/..amikom/6th/data-mining-project/artifacts/cbf_meta.pkl',
    'svd_model':      'D:/..amikom/6th/data-mining-project/artifacts/svd_model.pkl',
    'ratings_df':     'D:/..amikom/6th/data-mining-project/artifacts/ratings_filtered.pkl',
    'train_test':     'D:/..amikom/6th/data-mining-project/artifacts/train_test_split.pkl',
}

def load_artifacts():
    """
    Load all persisted artifacts. Call this at the top of any new session
    to skip retraining entirely.

    Returns True on success, False if any file is missing.
    """
    missing = [p for p in ARTIFACTS.values() if not os.path.exists(p)]
    if missing:
        print(f"Missing files (run save_artifacts() first): {missing}")
        return False

    global cbf_sim_matrix, cbf_df, cbf_title_to_idx, cbf_id_to_idx
    global svd_model, ratings_filtered, cf_all_book_ids, train_df, test_df

    # Reload as float64 for full precision during scoring
    cbf_sim_matrix = np.load(ARTIFACTS['cbf_sim_matrix']).astype(np.float64)

    with open(ARTIFACTS['cbf_meta'], 'rb') as f:
        meta = pickle.load(f)
    cbf_df           = meta['cbf_df']
    cbf_title_to_idx = meta['cbf_title_to_idx']
    cbf_id_to_idx    = meta['cbf_id_to_idx']

    with open(ARTIFACTS['svd_model'], 'rb') as f:
        svd_model = pickle.load(f)

    with open(ARTIFACTS['ratings_df'], 'rb') as f:
        rd = pickle.load(f)
    ratings_filtered = rd['ratings_filtered']
    cf_all_book_ids  = rd['cf_all_book_ids']

    with open(ARTIFACTS['train_test'], 'rb') as f:
        tt = pickle.load(f)
    train_df = tt['train_df']
    test_df  = tt['test_df']

    print("All artifacts loaded. Ready to serve recommendations.")
    return True

loaded = load_artifacts()


All artifacts loaded. Ready to serve recommendations.


## usage


In [45]:
print("\n" + "="*60)
print("HYBRID RECOMMENDATIONS")
print("="*60)

user_id    = 314
seed_title = "The Hunger Games (The Hunger Games, #1)"
alpha      = 0.8  # 0 = pure CBF, 1 = pure CF

recs = recommend_hybrid(user_id, seed_title, n=10, alpha=alpha)
print(f"Top 10 recommendations for user {user_id}")
print(f"Seed : '{seed_title}'")
print(f"Alpha (CF weight): {alpha}  |  CBF weight: {1-alpha}\n")
recs


HYBRID RECOMMENDATIONS
Top 10 recommendations for user 314
Seed : 'The Hunger Games (The Hunger Games, #1)'
Alpha (CF weight): 0.8  |  CBF weight: 0.19999999999999996



,Title,Authors,CF_Score,CBF_Score,Hybrid_Score
1,"Saga, Vol. 6 (Saga, #6)",author_brian_k_vaughan author_fiona_staples,0.9769,0.7479,0.9311
2,The Days Are Just Packed: A Calvin and Hobbes ...,author_bill_watterson,1.0000,0.5655,0.9131
3,The Complete Calvin and Hobbes,author_bill_watterson,0.9839,0.6179,0.9107
4,The Hunger Games Tribute Guide,author_emily_seife,0.8985,0.9523,0.9092
5,"Queen of Shadows (Throne of Glass, #4)",author_sarah_j_maas,0.9082,0.8876,0.9040
6,"Transmetropolitan, Vol. 5: Lonely City (Transm...",author_warren_ellis author_darick_robertson au...,0.9184,0.7873,0.8922
7,BookRags Summary: A Storm of Swords,author_bookrags,0.8935,0.8465,0.8841
8,"The Complete Maus (Maus, #1-2)",author_art_spiegelman,0.9482,0.6252,0.8836
9,"Saga, Vol. 5 (Saga, #5)",author_brian_k_vaughan author_fiona_staples,0.9210,0.7312,0.8831
10,"Saga, Vol. 1 (Saga, #1)",author_brian_k_vaughan author_fiona_staples,0.9018,0.8052,0.8825


In [11]:
print("\n" + "="*60)
print("HYBRID RECOMMENDATIONS")
print("="*60)

user_id    = 314
seed_title = "The Hunger Games (The Hunger Games, #1)"
alpha      = 0.8  # 0 = pure CBF, 1 = pure CF

recs = recommend_hybrid(user_id, seed_title, n=10, alpha=alpha)
print(f"Top 10 recommendations for user {user_id}")
print(f"Seed : '{seed_title}'")
print(f"Alpha (CF weight): {alpha}  |  CBF weight: {1-alpha}\n")
recs


HYBRID RECOMMENDATIONS
Top 10 recommendations for user 314
Seed : 'The Hunger Games (The Hunger Games, #1)'
Alpha (CF weight): 0.8  |  CBF weight: 0.19999999999999996



,Title,Authors,CF_Score,CBF_Score,Hybrid_Score
1,Peter and the Shadow Thieves (Peter and the St...,author_dave_barry author_ridley_pearson author...,0.8918,0.7938,0.8722
2,"American Gods (American Gods, #1)",author_neil_gaiman,0.8303,0.9471,0.8536
3,"The Salmon of Doubt (Dirk Gently, #3)",author_douglas_adams,0.8470,0.7774,0.8331
4,Still Life with Woodpecker,author_tom_robbins,0.8961,0.4988,0.8167
5,Prophet,author_frank_e_peretti,0.8641,0.6239,0.8161
6,Children of Dune (Dune Chronicles #3),author_frank_herbert,0.8133,0.8105,0.8128
7,"Three Men in a Boat (Three Men, #1)",author_jerome_k_jerome,0.8867,0.5158,0.8125
8,The Curious Incident of the Dog in the Night-Time,author_mark_haddon,0.8311,0.6950,0.8038
9,Deerskin,author_robin_mckinley,0.7766,0.9010,0.8015
10,The Days Are Just Packed: A Calvin and Hobbes ...,author_bill_watterson,1.0000,0.0000,0.8000


In [46]:
# ── Ranking evaluation ──────────────────────────────────────────────
summary, detail_df = print_evaluation_report(
    alpha=alpha,
    k_values=(5, 10, 20),
    relevance_threshold=4,   # ratings 4-5 = "relevant"
    n_test_users=200
)


  HYBRID RECOMMENDER — RANKING EVALUATION REPORT
  alpha=0.8  |  threshold≥4  |  users=200

Metric              @5       @10       @20
------------------------------------------
NDCG            0.0060    0.0046    0.0052
Precision       0.0064    0.0032    0.0021
Recall          0.0055    0.0055    0.0070
Hit Rate        0.0319    0.0319    0.0426
MAP             0.0025    0.0016    0.0016

Evaluated on 94 users.

Metric guide:
  NDCG      – ranking quality; rewards hits ranked higher (0→1, higher=better)
  Precision – fraction of top-k that are relevant
  Recall    – fraction of relevant items found in top-k
  Hit Rate  – % of users who got ≥1 relevant item in top-k
  MAP       – mean average precision across hit positions


In [47]:
# Optional: compare alpha values
print("\n── Alpha sensitivity (NDCG@10) ──")
for a in (0.0, 0.25, 0.5, 0.75, 1.0):
    s, _ = evaluate_hybrid(alpha=a, k_values=(10,), n_test_users=100)
    if s:
        print(f"  alpha={a:.2f}  NDCG@10={s['NDCG'][10]:.4f}  HR@10={s['HR'][10]:.4f}")


── Alpha sensitivity (NDCG@10) ──
  alpha=0.00  NDCG@10=0.0102  HR@10=0.0638
  alpha=0.25  NDCG@10=0.0118  HR@10=0.0638
  alpha=0.50  NDCG@10=0.0059  HR@10=0.0426
  alpha=0.75  NDCG@10=0.0023  HR@10=0.0213
  alpha=1.00  NDCG@10=0.0000  HR@10=0.0000
